## Widget

In [0]:
# Create a text widget for the catalog name, defaulting to 'adventureworks_dev'
dbutils.widgets.text("catalog", "adventureworks_dev")

# Retrieve the catalog name from the widget, fallback to 'adventureworks_dev' if not set
catalog_name = dbutils.widgets.get("catalog") or "adventureworks_dev"

## Catalog / Schema / Volume

In [0]:
# Create catalog, schemas, and volume for bronze landing if they do not exist
for ddl in [
    f"CREATE CATALOG  IF NOT EXISTS {catalog_name}",
    f"CREATE SCHEMA   IF NOT EXISTS {catalog_name}.bronze",
    f"CREATE VOLUME   IF NOT EXISTS {catalog_name}.bronze.landing",
]:
    spark.sql(ddl)


# Path to the landing directory for CSV files
landing_path = f"/Volumes/{catalog_name}/bronze/landing/csv"

# Create landing directory for CSV files in the bronze volume
dbutils.fs.mkdirs(landing_path)

##  Bronze table dictionary

In [0]:
# All columns STRING — types are cast in the silver layer
tables = {
    # ── SALES ────────────────────────────────────────────────────────────────
    "Customer": [
        "CustomerID",
        "PersonID",
        "StoreID",
        "TerritoryID",
        "AccountNumber",
        "rowguid",
        "ModifiedDate",
    ],
    "SalesOrderHeader": [
        "SalesOrderID",
        "RevisionNumber",
        "OrderDate",
        "DueDate",
        "ShipDate",
        "Status",
        "OnlineOrderFlag",
        "SalesOrderNumber",
        "PurchaseOrderNumber",
        "AccountNumber",
        "CustomerID",
        "SalesPersonID",
        "TerritoryID",
        "BillToAddressID",
        "ShipToAddressID",
        "ShipMethodID",
        "CreditCardID",
        "CreditCardApprovalCode",
        "CurrencyRateID",
        "SubTotal",
        "TaxAmt",
        "Freight",
        "TotalDue",
        "Comment",
        "rowguid",
        "ModifiedDate",
    ],
    "SalesOrderDetail": [
        "SalesOrderID",
        "SalesOrderDetailID",
        "CarrierTrackingNumber",
        "OrderQty",
        "ProductID",
        "SpecialOfferID",
        "UnitPrice",
        "UnitPriceDiscount",
        "LineTotal",
        "rowguid",
        "ModifiedDate",
    ],
    "SalesTerritory": [
        "TerritoryID",
        "Name",
        "CountryRegionCode",
        "Group",
        "SalesYTD",
        "SalesLastYear",
        "CostYTD",
        "CostLastYear",
        "rowguid",
        "ModifiedDate",
    ],
    "SalesTerritoryHistory": [
        "BusinessEntityID",
        "TerritoryID",
        "StartDate",
        "EndDate",
        "rowguid",
        "ModifiedDate",
    ],
    "SalesPerson": [
        "BusinessEntityID",
        "TerritoryID",
        "SalesQuota",
        "Bonus",
        "CommissionPct",
        "SalesYTD",
        "SalesLastYear",
        "rowguid",
        "ModifiedDate",
    ],
    "SpecialOffer": [
        "SpecialOfferID",
        "Description",
        "DiscountPct",
        "Type",
        "Category",
        "StartDate",
        "EndDate",
        "MinQty",
        "MaxQty",
        "rowguid",
        "ModifiedDate",
    ],
    "SpecialOfferProduct": [
        "SpecialOfferID",
        "ProductID",
        "rowguid",
        "ModifiedDate",
    ],
    "CurrencyRate": [
        "CurrencyRateID",
        "CurrencyRateDate",
        "FromCurrencyCode",
        "ToCurrencyCode",
        "AverageRate",
        "EndOfDayRate",
        "ModifiedDate",
    ],
    "ShipMethod": [
        "ShipMethodID",
        "Name",
        "ShipBase",
        "ShipRate",
        "rowguid",
        "ModifiedDate",
    ],
    "CreditCard": [
        "CreditCardID",
        "CardType",
        "CardNumber",
        "ExpMonth",
        "ExpYear",
        "ModifiedDate",
    ],
    # ── HUMAN RESOURCES ──────────────────────────────────────────────────────
    "Employee": [
        "BusinessEntityID",
        "NationalIDNumber",
        "LoginID",
        "OrganizationNode",
        "OrganizationLevel",
        "JobTitle",
        "BirthDate",
        "MaritalStatus",
        "Gender",
        "HireDate",
        "SalariedFlag",
        "VacationHours",
        "SickLeaveHours",
        "CurrentFlag",
        "rowguid",
        "ModifiedDate",
    ],
    "EmployeePayHistory": [
        "BusinessEntityID",
        "RateChangeDate",
        "Rate",
        "PayFrequency",
        "ModifiedDate",
    ],
    "EmployeeDepartmentHistory": [
        "BusinessEntityID",
        "DepartmentID",
        "ShiftID",
        "StartDate",
        "EndDate",
        "ModifiedDate",
    ],
    # ── PERSON ───────────────────────────────────────────────────────────────
    "BusinessEntityAddress": [
        "BusinessEntityID",
        "AddressID",
        "AddressTypeID",
        "rowguid",
        "ModifiedDate",
    ],
    "Person": [
        "BusinessEntityID",
        "PersonType",
        "NameStyle",
        "Title",
        "FirstName",
        "MiddleName",
        "LastName",
        "Suffix",
        "EmailPromotion",
        "AdditionalContactInfo",
        "Demographics",
        "rowguid",
        "ModifiedDate",
    ],
    "PersonPhone": [
        "BusinessEntityID",
        "PhoneNumber",
        "PhoneNumberTypeID",
        "ModifiedDate",
    ],
    "EmailAddress": [
        "BusinessEntityID",
        "EmailAddressID",
        "EmailAddress",
        "rowguid",
        "ModifiedDate",
    ],
    "Address": [
        "AddressID",
        "AddressLine1",
        "AddressLine2",
        "City",
        "StateProvinceID",
        "PostalCode",
        "SpatialLocation",
        "rowguid",
        "ModifiedDate",
    ],
    "StateProvince": [
        "StateProvinceID",
        "StateProvinceCode",
        "CountryRegionCode",
        "IsOnlyStateProvinceFlag",
        "Name",
        "TerritoryID",
        "rowguid",
        "ModifiedDate",
    ],
    # ── PRODUCTION ───────────────────────────────────────────────────────────
    "Product": [
        "ProductID",
        "Name",
        "ProductNumber",
        "MakeFlag",
        "FinishedGoodsFlag",
        "Color",
        "SafetyStockLevel",
        "ReorderPoint",
        "StandardCost",
        "ListPrice",
        "Size",
        "SizeUnitMeasureCode",
        "WeightUnitMeasureCode",
        "Weight",
        "DaysToManufacture",
        "ProductLine",
        "Class",
        "Style",
        "ProductSubcategoryID",
        "ProductModelID",
        "SellStartDate",
        "SellEndDate",
        "DiscontinuedDate",
        "rowguid",
        "ModifiedDate",
    ],
    "ProductSubcategory": [
        "ProductSubcategoryID",
        "ProductCategoryID",
        "Name",
        "rowguid",
        "ModifiedDate",
    ],
    "ProductCategory": [
        "ProductCategoryID",
        "Name",
        "rowguid",
        "ModifiedDate",
    ],
    "ProductListPriceHistory": [
        "ProductID",
        "StartDate",
        "EndDate",
        "ListPrice",
        "ModifiedDate",
    ],
    "ProductInventory": [
        "ProductID",
        "LocationID",
        "Shelf",
        "Bin",
        "Quantity",
        "rowguid",
        "ModifiedDate",
    ],
}

## Download CSVs from GitHub

In [0]:
import requests

BASE_URL = (
    "https://raw.githubusercontent.com/microsoft/sql-server-samples"
    "/master/samples/databases/adventure-works/oltp-install-script"
)

failed = []

for table in tables:
    url = f"{BASE_URL}/{table}.csv"
    dest = f"{landing_path}/{table}.csv"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(dest, "wb") as f:
            f.write(response.content)

        print(f"  ✓  {table}.csv  ({len(response.content):,} bytes)")

    except Exception as e:
        failed.append(table)
        print(f"  ✗  {table}.csv  — {e}")

if failed:
    raise Exception(
        f"Download failed for {len(failed)} file(s):\n  - " + "\n  - ".join(failed)
    )

print(f"\nDownloaded all {len(tables)} CSV files to {landing_path}")

## Pre-flight check

In [0]:
expected = {f"{table}.csv" for table in tables}

try:
    found = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    raise Exception(
        f"Pre-flight failed: cannot list landing path '{landing_path}'.\n"
        f"  Make sure the catalog '{catalog_name}' and Volume exist.\n"
        f"Underlying error: {e}"
    )

missing = sorted(expected - found)
if missing:
    raise Exception(
        f"Pre-flight failed: {len(missing)} of {len(expected)} CSV files are "
        f"MISSING from\n  {landing_path}\n\n"
        "Missing files:\n  - " + "\n  - ".join(missing) + "\n\n"
        "ACTION REQUIRED:\n"
        "  1. Download the AdventureWorks OLTP CSV exports:\n"
        "     https://github.com/microsoft/sql-server-samples/tree/master/"
        "samples/databases/adventure-works/oltp-install-script\n"
        f"  2. Upload the missing .csv files to {landing_path}\n"
        "  3. Re-run this task."
    )

print(f"Pre-flight passed — all {len(expected)} CSV files found in {landing_path}")

## Load CSVs

In [0]:
from pyspark.sql.types import StructType, StructField, StringType


def load_csv(table, sep, multiline=False):
    columns = [c.strip("`") for c in tables[table]]
    schema = StructType([StructField(c, StringType(), True) for c in columns])

    df = (
        spark.read.option("header", False)
        .option("sep", sep)
        .option("encoding", "UTF-8")
        .option("nullValue", "")
        .option("multiLine", multiline)
        .option("escape", '"')
        .schema(schema)
        .csv(f"{landing_path}/{table}.csv")
    )

    df.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze.{table}")


# Tab-delimited
tab_delimited = [
    "Customer",
    "SalesOrderHeader",
    "SalesOrderDetail",
    "SalesTerritory",
    "SalesTerritoryHistory",
    "SalesPerson",
    "SpecialOffer",
    "SpecialOfferProduct",
    "CurrencyRate",
    "ShipMethod",
    "CreditCard",
    "Employee",
    "EmployeePayHistory",
    "EmployeeDepartmentHistory",
    "Address",
    "StateProvince",
    "Product",
    "ProductSubcategory",
    "ProductCategory",
    "ProductListPriceHistory",
    "ProductInventory",
]
pipe_delimited = ["BusinessEntityAddress", "Person", "PersonPhone", "EmailAddress"]

for table in tab_delimited:
    load_csv(table, sep="\t")

for table in pipe_delimited:
    load_csv(table, sep="+|", multiline=True)